In [12]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [13]:
load_dotenv()

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [14]:
class SentimentSchema(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(description="Sentiment of the review.")
    

In [15]:
class DiagnosisSchema(BaseModel):
    issue: Literal["performance", "bug", "support", "other"] = Field(
        description="The category of the customer's issue"
    )
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(
        description="The emotional tone of the review"
    )
    urgency: Literal["low", "medium", "high"] = Field(
        description="How urgent or critical the issue is"
    )

In [16]:
structured_model= model.with_structured_output(SentimentSchema)
structured_model2= model.with_structured_output(DiagnosisSchema)


In [17]:
class ReviewState(TypedDict):
    
    review: str
    sentiment: Literal["positive", "negative"]
    diagnosis: dict
    response: str

In [28]:
def find_sentiment(state: ReviewState):
    prompt=f'For the following review, find the sentiment of the review. If the review is positive, return "positive". If the review is negative, return "negative". Review: {state["review"]}'
    sentiment = structured_model.invoke(prompt).sentiment
    state["sentiment"] = sentiment
    
    return {'sentiment': sentiment}

def check_sentiment(state: ReviewState) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    return "run_diagnosis"
    
def positive_response(state: ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
{state["review"]}
Also, kindly ask the reviewer to leave feedback on our website."""

    response = model.invoke(prompt).content
    return {"response": response}

def run_diagnosis(state: ReviewState):
    prompt = f"""Diagnose this negative review:\n{state['review']}\n
    Return issue_type, tone, and urgency."""

    response = structured_model2.invoke(prompt)
    return {'diagnosis': response.model_dump()}

def negetive_response(state: ReviewState):
    prompt = f"""You are a support assistant.
The user has a "{state['diagnosis']['issue']}" issue, sounded "{state['diagnosis']['tone']}",
and marked urgency "{state['diagnosis']['urgency']}".
Write an empathetic, helpful resolution message."""

    response = model.invoke(prompt).content
    return {'response': response}

In [29]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
#graph.add_node('check_sentiment', check_sentiment)
graph.add_node('positive_response', positive_response)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('negetive_response', negetive_response)





graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_sentiment)

graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negetive_response')
graph.add_edge('negetive_response', END)

workflow = graph.compile()


In [31]:
initial_state = {'review': 'The product was bad bad bad!', 'sentiment': ''}

workflow.invoke(initial_state)

{'review': 'The product was bad bad bad!',
 'sentiment': 'negative',
 'diagnosis': {'issue': 'other', 'tone': 'angry', 'urgency': 'medium'},
 'response': 'Subject: Re: Your Recent Issue - We\'re Here to Help\n\nDear [User\'s Name, if known, otherwise "Customer"],\n\nI\'m truly sorry to hear that you\'re experiencing a frustrating situation, and I understand you\'re feeling upset about it. My priority is to help you get this resolved quickly and efficiently.\n\nTo ensure I can provide the best possible assistance, could you please tell me a bit more about what happened? Since your issue is categorized as \'other,\' any details you can provide would be incredibly helpful, such as:\n\n*   **What exactly is the problem you\'re encountering?**\n*   **When did you first notice it?**\n*   **What were you trying to do when it occurred?**\n*   **Are there any error messages or specific symptoms?**\n*   **What impact is this having on you?**\n\nI\'ve noted your urgency as \'medium,\' and I assur